[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IgnatiusEzeani/spatial-humanities-2026/blob/main/workshop/09_responsible_spatial_ai.ipynb)

# 09 · Responsible Spatial AI: provenance, uncertainty and release audit

**Spatial Humanities 2026 workshop**

The final notebook turns the methodological lessons of the day into a practical audit framework. The question is no longer only *"Can the system extract this?"* but also *"Can another scholar understand, verify and responsibly reuse what the system produced?"*

## Learning goals
- create a secret-safe reproducibility manifest;
- identify provenance and uncertainty requirements for each pipeline stage;
- distinguish public, restricted and sensitive materials;
- document model/provider/version drift;
- design human-review triggers;
- evaluate cost, latency and portability alongside accuracy;
- use a release checklist before sharing notebooks, demos or benchmark results.

> **Key message:** The more interpretive power we delegate to AI, the stronger the audit trail must become.

In [1]:
!wget -qO sh2026_setup.py https://raw.githubusercontent.com/IgnatiusEzeani/spatial-humanities-2026/main/workshop/sh2026_setup.py

import sh2026_setup as sh
ctx = sh.setup()

# Bound from the shared context: the cells below were written against these.
repo_dir = ctx.repo
data_dir = ctx.data

import json
REPO = "https://github.com/IgnatiusEzeani/spatio-textual.git"


Reusing existing checkout at /home/ezeani/workspace/spatial-humanities-2026
Dependencies already installed in this runtime.

Ready in 0s.
  repo    : /home/ezeani/workspace/spatial-humanities-2026
  commit  : 798f2be
  data    : /home/ezeani/workspace/spatial-humanities-2026/workshop/data
  outputs : /home/ezeani/workspace/spatial-humanities-2026/sh2026_outputs
  route   : CPU only, no API key needed

If this cell failed, put your hand up. Do not re-run it more than once.


## 1. Reproducibility begins with a run manifest

A useful research output should record enough information to understand how it was produced without exposing credentials.

In [2]:
from spatio_textual.provenance import build_run_manifest

example_config = {
    "task": "journey_extraction",
    "backend": "llm",
    "provider": "example-provider",
    "model": "example-model",
    "temperature": 0,
    "api_key": "this-value-must-never-be-exported",
}

manifest = build_run_manifest(
    input_text="I travelled from Cambridge to London.",
    config=example_config,
    git_commit="spatial-humanities-2026",
)

print(json.dumps(manifest, indent=2, ensure_ascii=False))

{
  "timestamp": "2026-09-20T19:47:06.930701+00:00",
  "package": "spatio-textual",
  "package_version": "0.4.1",
  "git_commit": "spatial-humanities-2026",
  "python_version": "3.12.3",
  "platform": "Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39",
  "input_sha256": "8f1839fb6b0cc699da0e0753b056dccfa682d26bb5e1efd2b602b31368ea9923",
  "input_chars": 37,
  "config": {
    "task": "journey_extraction",
    "backend": "llm",
    "provider": "example-provider",
    "model": "example-model",
    "temperature": 0,
    "api_key": "<redacted>"
  }
}


The manifest stores an **input hash**, not the source text itself. That is useful when data cannot be redistributed: a researcher can still verify whether two runs used identical input without exposing the input in the manifest.

Secret-like configuration keys are redacted automatically.

## 2. Audit every transformation, not only the final model

The SH2026 pipeline can be read as a chain of interpretive transformations:

**source → segmentation → recognition → resolution → relation/event → LLM structure → adjudication → map/visualisation → interpretation**

At each step ask:

1. What input was used?
2. What transformation occurred?
3. What assumptions or ontology constrained the output?
4. What uncertainty was introduced?
5. What evidence/provenance remains available?
6. What can a human correct?
7. What gets lost if we export only the final result?

In [3]:
import pandas as pd

audit_matrix = pd.DataFrame([
    ["Segmentation", "sentence/turn boundaries", "lost context", "segment offsets", "inspect/resegment"],
    ["Recognition", "label inventory + model", "missed/mislabelled spans", "source offsets", "accept/edit/reject"],
    ["Resolution", "gazetteer + ranking", "ambiguity/anachronism", "candidate list + source string", "human resolution"],
    ["Affect", "taxonomy + backend", "domain/ontology bias", "distribution + backend", "reinterpret/reclassify"],
    ["Journey extraction", "schema + prompt + LLM", "unsupported inference", "verbatim quote + offsets", "field-level review"],
    ["Mapping", "coordinate/geometry choices", "false precision/omissions", "GeoJSON + audit", "exclude/flag/remap"],
], columns=["stage", "assumption", "risk", "minimum_provenance", "human_action"])

display(audit_matrix)

,stage,assumption,risk,minimum_provenance,human_action
0,Segmentation,sentence/turn boundaries,lost context,segment offsets,inspect/resegment
1,Recognition,label inventory + model,missed/mislabelled spans,source offsets,accept/edit/reject
2,Resolution,gazetteer + ranking,ambiguity/anachronism,candidate list + source string,human resolution
3,Affect,taxonomy + backend,domain/ontology bias,distribution + backend,reinterpret/reclassify
4,Journey extraction,schema + prompt + LLM,unsupported inference,verbatim quote + offsets,field-level review
5,Mapping,coordinate/geometry choices,false precision/omissions,GeoJSON + audit,exclude/flag/remap


## 3. Uncertainty must survive export

A common failure mode is to show uncertainty in an interface but export only the clean final label or coordinate.

For SH2026, uncertainty belongs in the data model:

- `requires_review`
- `review_notes`
- `ambiguous`
- candidate lists
- `explicit_or_inferred`
- `evidence_grounded`
- vote ratios/disagreements
- human status/edit trail

If uncertainty disappears at export time, the research object becomes less trustworthy than the interface that produced it.

## 4. Public, restricted and sensitive material

The workshop/demo release must separate **code openness** from **data openness**.

### Safe to publish when rights permit
- package code;
- schemas;
- synthetic teaching examples;
- public-domain or appropriately licensed source excerpts;
- aggregate statistics;
- benchmark methodology;
- precomputed outputs derived from distributable inputs.

### Do not publish merely because the code repository is public
- controlled-access transcripts;
- donor/private materials;
- API keys and credentials;
- `.env` files;
- data whose licence prohibits redistribution;
- raw provider logs containing sensitive source text.

For controlled collections, a public notebook can still demonstrate the **method** using synthetic or cleared examples while keeping restricted inputs outside the repository.

## 5. Model and provider drift

A model name is not always a permanent scientific object. Hosted models may change, aliases may point to newer revisions, SDK behaviour can change and APIs can be retired.

For any result used in the keynote or paper, record where available:

- provider;
- exact model identifier;
- model revision/date if exposed;
- package/SDK version;
- prompt/schema version;
- inference parameters;
- timestamp;
- Git commit;
- latency/token telemetry;
- whether output was live or precomputed.

If exact reproducibility is impossible, document the limitation rather than implying otherwise.

## 6. Human review triggers

Human review should not be an informal afterthought. Define triggers before running the benchmark.

In [4]:
review_triggers = pd.DataFrame([
    ["Entity resolution", "multiple plausible candidates", "ambiguous_place"],
    ["Entity resolution", "no safe modern resolution for historical polity", "unresolved_place"],
    ["NER / MoE", "models disagree", "model_disagreement"],
    ["Journey", "any contextual inference", "contextual_inference"],
    ["Journey", "evidence quote not grounded", "unsupported_llm_field"],
    ["Any model", "backend error or invalid schema", "backend_error"],
    ["Human", "researcher notices interpretive problem", "human_flag"],
], columns=["component", "trigger", "review_reason"])

display(review_triggers)

,component,trigger,review_reason
0,Entity resolution,multiple plausible candidates,ambiguous_place
1,Entity resolution,no safe modern resolution for historical polity,unresolved_place
2,NER / MoE,models disagree,model_disagreement
3,Journey,any contextual inference,contextual_inference
4,Journey,evidence quote not grounded,unsupported_llm_field
5,Any model,backend error or invalid schema,backend_error
6,Human,researcher notices interpretive problem,human_flag


A review queue is therefore a **research instrument**: it reveals where automation encounters interpretive difficulty.

## 7. Evaluate more than accuracy

For the keynote benchmark, accuracy is necessary where a gold/reference task exists, but it is not sufficient.

In [5]:
benchmark_dimensions = pd.DataFrame([
    ["Accuracy", "precision / recall / F1 where appropriate"],
    ["Representational reach", "which spatial phenomena the method can express"],
    ["Unsupported inference", "claims without grounded source evidence"],
    ["Ambiguity handling", "whether uncertainty is surfaced or silently collapsed"],
    ["Human review burden", "fraction of records requiring inspection"],
    ["Human correction burden", "fraction edited/rejected after inspection"],
    ["Latency", "time per item / per corpus"],
    ["Compute/API cost", "documented estimated cost where measurable"],
    ["Portability", "effort to adapt to another corpus/domain"],
    ["Reproducibility", "determinism, versions, prompts, provenance"],
    ["Governance", "privacy, data transfer, provider dependence"],
], columns=["dimension", "question"])

display(benchmark_dimensions)

,dimension,question
0,Accuracy,precision / recall / F1 where appropriate
1,Representational reach,which spatial phenomena the method can express
2,Unsupported inference,claims without grounded source evidence
3,Ambiguity handling,whether uncertainty is surfaced or silently co...
4,Human review burden,fraction of records requiring inspection
5,Human correction burden,fraction edited/rejected after inspection
6,Latency,time per item / per corpus
7,Compute/API cost,documented estimated cost where measurable
8,Portability,effort to adapt to another corpus/domain
9,Reproducibility,"determinism, versions, prompts, provenance"


This prevents the comparison from collapsing into a single leaderboard number.

## 8. Release audit checklist

Run this before publishing the workshop/demo release candidate.

In [6]:
release_checks = {
    "all_CI_green": False,
    "python_compatibility_green": False,
    "all_notebooks_run_in_fresh_colab": False,
    "no_credentials_or_env_files": False,
    "no_controlled_access_text": False,
    "teaching_source_citations_verified": False,
    "heldout_benchmark_frozen_before_final_tuning": False,
    "precomputed_outputs_have_model_version_metadata": False,
    "demo_tested_from_clean_browser": False,
    "uncertainty_preserved_in_exports": False,
    "official_titles_and_terminology_synchronised": False,
    "fallback_plan_tested_without_GPU_or_API": False,
}

pd.Series(release_checks, name="complete").to_frame()

,complete
all_CI_green,False
python_compatibility_green,False
all_notebooks_run_in_fresh_colab,False
no_credentials_or_env_files,False
no_controlled_access_text,False
teaching_source_citations_verified,False
heldout_benchmark_frozen_before_final_tuning,False
precomputed_outputs_have_model_version_metadata,False
demo_tested_from_clean_browser,False
uncertainty_preserved_in_exports,False


The release candidate is not ready because a notebook *looks finished*. It is ready when these gates have evidence behind them.

## 9. Final reusable questions

Before accepting any AI-generated spatial representation, ask:

1. **What exactly is being claimed?**
2. **Where is the supporting source evidence?**
3. **Is the claim explicit, inferred, ambiguous or missing?**
4. **Which model/rule/gazetteer produced it?**
5. **What assumptions does that method encode?**
6. **Could historical geography make a modern coordinate misleading?**
7. **What did the system omit because it could not map/label it?**
8. **Can a human inspect and correct the result?**
9. **Will that correction remain auditable?**
10. **Could another researcher reproduce or at least understand this run?**
11. **Does the data licence/privacy context permit this processing route?**
12. **Would a null/unresolved answer be more defensible than a guessed one?**

## 10. Closing proposition

Across the workshop we moved from manual annotation to rules, contextual NLP, LLM extraction, adjudication and mapping. The central lesson is not that one method replaces the others.

It is this:

> **Increasing representational capability shifts the bottleneck from extraction toward validation, provenance, interpretation and governance.**

That proposition now connects the tutorial directly to the keynote: *From Coordinates to Context: Rethinking Spatial Humanities in the Age of Large Language Models*.